# SwiftAlign — Alignment Training Demo

> **Compact DPO / GRPO scaffold** for Qwen2.5-1.5B-Instruct and similar small models.
> Designed for Colab T4 (15 GB VRAM) with optional QLoRA for tighter environments.

### What this notebook does
1. Installs dependencies
2. Runs hardware detection
3. Loads the model with optional LoRA/QLoRA
4. Trains with DPO **or** GRPO on a built-in synthetic dataset
5. Generates a sample completion from the trained model

---

In [ ]:
# ── Cell 1 · Install dependencies ────────────────────────────────────────────
# Run once per Colab session.

!pip install -q \
    transformers>=4.40 \
    trl>=0.8 \
    peft>=0.10 \
    bitsandbytes>=0.43 \
    datasets>=2.18 \
    accelerate>=0.28 \
    rich

# Clone the SwiftAlign repo (or mount your Drive copy)
import os
if not os.path.exists('swiftalign'):
    !git clone https://github.com/NovatasticRoScript/swiftalign.git

%cd swiftalign
print('Ready.')

In [ ]:
# ── Cell 2 · Hardware check ───────────────────────────────────────────────────
from swiftalign.hardware import detect_hardware
from swiftalign.dashboard import Dashboard

dash = Dashboard(enabled=True)
dash.banner()

hw = detect_hardware()
for note in hw.extra_notes:
    print(' ·', note)
dash.hardware_table(hw.summary())

In [ ]:
# ── Cell 3 · Configuration ────────────────────────────────────────────────────
# Edit these values before running.

import types

args = types.SimpleNamespace(
    method      = 'dpo',                            # 'dpo' or 'grpo'
    model       = 'Qwen/Qwen2.5-1.5B-Instruct',    # any causal LM on HF Hub
    lora        = False,
    qlora       = True,                             # recommended for T4
    output_dir  = './output',
    epochs      = 1,
    batch_size  = 2,
    max_steps   = 20,  # set -1 to run full epoch
    lr          = 5e-5,
    seed        = 42,
    no_dashboard= False,
)

print(f'Method: {args.method} | Model: {args.model}')
print(f'QLoRA: {args.qlora} | Max steps: {args.max_steps}')

In [ ]:
# ── Cell 4 · Run training ─────────────────────────────────────────────────────
from swiftalign.runner import run

metrics = run(args)
print('\nFinal metrics:', metrics)

In [ ]:
# ── Cell 5 · Sample generation from the trained model ─────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

output_dir = args.output_dir
print(f'Loading trained model from {output_dir} ...')

tok = AutoTokenizer.from_pretrained(output_dir, trust_remote_code=True)

try:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(
        args.model,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    model_inf = PeftModel.from_pretrained(base, output_dir)
    print('Loaded as PeftModel (LoRA/QLoRA adapter)')
except Exception:
    model_inf = AutoModelForCausalLM.from_pretrained(
        output_dir,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    print('Loaded as merged model')

model_inf.eval()

prompt = 'Explain the difference between DPO and GRPO alignment methods.'
inputs = tok(prompt, return_tensors='pt').to(model_inf.device)

with torch.no_grad():
    outputs = model_inf.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tok.eos_token_id,
    )

response = tok.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('\n' + '='*60)
print('PROMPT:', prompt)
print('-'*60)
print('RESPONSE:', response)
print('='*60)

In [ ]:
# ── Cell 6 · Reward functions demo ───────────────────────────────────────────
from swiftalign.rewards import (
    reward_length_penalty,
    reward_format_check,
    reward_keyword_presence,
    combined_reward,
)

prompts = ['Explain gradient descent in neural networks.']
completions = [
    'Gradient descent is an optimisation algorithm that updates model weights '
    'by moving in the direction of steepest loss decrease, scaled by the learning rate.',
]

print('length_penalty:', reward_length_penalty(prompts, completions))
print('format_check:  ', reward_format_check(prompts, completions))
print('keyword:       ', reward_keyword_presence(prompts, completions))
print('combined:      ', combined_reward(prompts, completions))